# Обучение головы регрессии рейтинга SoFIFA по агрегированным эмбеддингам

Агрегируем эмбеддинги игроков **по сезону** (по всем матчам сезона усредняем выход энкодера по позиции игрока), затем обучаем голову предсказания **overall** (данные из `dataset/sofifa_players.csv`). Энкодер заморожен. Голова сохраняется в `rating_head_output/rating_head.pt`.

In [1]:
import sys
from pathlib import Path
import pickle

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from safetensors.torch import load_file as load_safetensors

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
ENCODER_CKPT_DIR = ROOT / "model_trained" / "embed_128"
ENCODER_CKPT_DIR_64 = ROOT / "model_trained" / "embed_64"
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
PROCESSED_CSV = ROOT / "notebooks" / "train_sample_processed" / "processed.csv"
SOFIFA_CSV = ROOT / "dataset" / "sofifa_players.csv"
OUTPUT_DIR = ROOT / "notebooks" / "rating_head_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embed_64").mkdir(parents=True, exist_ok=True)

assert ENCODER_CKPT_DIR.exists(), f"Чекпоинт не найден: {ENCODER_CKPT_DIR}"
assert ENCODER_CKPT_DIR_64.exists(), f"Чекпоинт 64 не найден: {ENCODER_CKPT_DIR_64}"
assert METADATA_DIR.exists(), f"Метаданные не найдены: {METADATA_DIR}"
assert PROCESSED_CSV.exists(), f"CSV не найден: {PROCESSED_CSV}"
assert SOFIFA_CSV.exists(), f"SoFIFA CSV не найден: {SOFIFA_CSV}"

In [3]:
def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n = len(player_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n + 1,
        "team_pad_token_id": len(team_name2id),
    }

vocab = load_vocab(METADATA_DIR)
player_pad_token_id = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1
print("player_pad_token_id:", player_pad_token_id)

player_pad_token_id: 6394


In [4]:
import pandas as pd
from data.dataset import MatchDatasetMPP
from data.sofifa import load_sofifa_csv, build_player_id_to_overall, build_aggregated_embeddings, SofifaAggregatedDataset
from models.encoder import PlayerEncoder
from data.preprocessing import FORM_STATS_SIZE

df = pd.read_csv(PROCESSED_CSV)
match_ds = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=36,
    player_pad_token_id=player_pad_token_id,
    team_pad_token_id=vocab["team_pad_token_id"],
)

sofifa_df = load_sofifa_csv(SOFIFA_CSV)
player_id_to_overall = build_player_id_to_overall(sofifa_df, vocab["player_name2id"], max_id=player_pad_token_id - 1)
print("Игроков в SoFIFA со словарём:", len(player_id_to_overall))

embed_size = 128
encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state = load_safetensors(ENCODER_CKPT_DIR / "model.safetensors")
enc_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(enc_state, strict=True)
for p in encoder.parameters():
    p.requires_grad = False

print("Строим агрегированные эмбеддинги по сезонам...")
embeddings, meta = build_aggregated_embeddings(
    encoder, match_ds, df, player_id_to_overall, device
)
print("Сэмплов (player, season) с overall:", len(meta))

Игроков в SoFIFA со словарём: 1541
Строим агрегированные эмбеддинги по сезонам...
Сэмплов (player, season) с overall: 2814


In [5]:
agg_ds = SofifaAggregatedDataset(embeddings, meta)
n = len(agg_ds)
n_val = max(1, int(n * 0.15))
n_train = n - n_val
train_ds = Subset(agg_ds, range(0, n_train))
eval_ds = Subset(agg_ds, range(n_train, n))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, num_workers=0)
print("Train:", n_train, "Eval:", n_val)

Train: 2392 Eval: 422


In [6]:
from models.heads import RegressionHead
from training.rating_trainer import RatingHeadTrainer

head = RegressionHead(embed_size=embed_size, output_dim=1, hidden_dim=128, pool="per_sequence")
head = head.to(device)

trainer = RatingHeadTrainer(
    head=head,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR),
    num_epochs=100,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=10,
    save_best=True,
)
trainer.train()

Epoch 1/100  train_loss=4872.5994  eval_rmse=58.9309
Epoch 10/100  train_loss=277.8756  eval_rmse=17.5004
Epoch 20/100  train_loss=95.6653  eval_rmse=12.2322
Epoch 30/100  train_loss=56.0004  eval_rmse=10.8440
Epoch 40/100  train_loss=43.9991  eval_rmse=10.3681
Epoch 50/100  train_loss=35.8084  eval_rmse=10.0654
Epoch 60/100  train_loss=32.0323  eval_rmse=9.9195
Epoch 70/100  train_loss=29.8688  eval_rmse=9.6812
Epoch 80/100  train_loss=27.5731  eval_rmse=9.5989
Epoch 90/100  train_loss=26.5323  eval_rmse=9.6334
Epoch 100/100  train_loss=22.3993  eval_rmse=9.6899
Best eval RMSE: 9.478556111921433
Голова сохранена: C:\Users\vasilii\Documents\ML_project-football-\notebooks\rating_head_output\rating_head.pt


9.478556111921433

In [7]:
# Энкодер 64 и голова 64: агрегированные эмбеддинги + обучение головы
embed_size_64 = 64
encoder_64 = PlayerEncoder(
    embed_size=embed_size_64,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state_64 = load_safetensors(ENCODER_CKPT_DIR_64 / "model.safetensors")
enc_state_64 = {k.replace("encoder.", ""): v for k, v in state_64.items() if k.startswith("encoder.")}
encoder_64.load_state_dict(enc_state_64, strict=True)
for p in encoder_64.parameters():
    p.requires_grad = False

print("Строим агрегированные эмбеддинги (embed_64)...")
embeddings_64, _ = build_aggregated_embeddings(
    encoder_64, match_ds, df, player_id_to_overall, device
)
agg_ds_64 = SofifaAggregatedDataset(embeddings_64, meta)
train_ds_64 = Subset(agg_ds_64, range(0, n_train))
eval_ds_64 = Subset(agg_ds_64, range(n_train, n))
train_loader_64 = DataLoader(train_ds_64, batch_size=64, shuffle=True, num_workers=0)
eval_loader_64 = DataLoader(eval_ds_64, batch_size=64, shuffle=False, num_workers=0)

head_64 = RegressionHead(embed_size=embed_size_64, output_dim=1, hidden_dim=128, pool="per_sequence")
head_64 = head_64.to(device)
trainer_64 = RatingHeadTrainer(
    head=head_64,
    train_loader=train_loader_64,
    eval_loader=eval_loader_64,
    output_dir=str(OUTPUT_DIR / "embed_64"),
    num_epochs=100,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=10,
    save_best=True,
)
trainer_64.train()
print("Голова 64 сохранена:", OUTPUT_DIR / "embed_64" / "rating_head.pt")

Строим агрегированные эмбеддинги (embed_64)...
Epoch 1/100  train_loss=4874.8081  eval_rmse=61.0592
Epoch 10/100  train_loss=237.2035  eval_rmse=15.3115
Epoch 20/100  train_loss=92.4371  eval_rmse=10.9073
Epoch 30/100  train_loss=66.0022  eval_rmse=9.9878
Epoch 40/100  train_loss=53.4326  eval_rmse=9.7124
Epoch 50/100  train_loss=48.4201  eval_rmse=9.4633
Epoch 60/100  train_loss=41.1324  eval_rmse=9.3068
Epoch 70/100  train_loss=39.4318  eval_rmse=9.1436
Epoch 80/100  train_loss=34.0796  eval_rmse=9.1268
Epoch 90/100  train_loss=34.0679  eval_rmse=9.1206
Epoch 100/100  train_loss=31.1742  eval_rmse=9.0898
Best eval RMSE: 8.98587894839843
Голова сохранена: C:\Users\vasilii\Documents\ML_project-football-\notebooks\rating_head_output\embed_64\rating_head.pt
Голова 64 сохранена: C:\Users\vasilii\Documents\ML_project-football-\notebooks\rating_head_output\embed_64\rating_head.pt


In [10]:
# Таблица: имя игрока, настоящий рейтинг SoFIFA, предсказание 128, предсказание 64 (на eval)
player_id_to_name = {pid: name for name, pid in vocab["player_name2id"].items()}

head.eval()
head_64.eval()
pids_list, names_list, trues, preds_128, preds_64 = [], [], [], [], []
with torch.no_grad():
    for idx in range(n_train, n):
        row = meta.iloc[idx]
        pid = row["player_id"]
        name = player_id_to_name.get(pid, "?")
        overall = row["overall"]
        emb_128 = torch.from_numpy(embeddings[idx].copy()).float().unsqueeze(0).unsqueeze(1).to(device)
        emb_64 = torch.from_numpy(embeddings_64[idx].copy()).float().unsqueeze(0).unsqueeze(1).to(device)
        mask = torch.ones(1, 1, dtype=torch.long, device=device)
        p128 = head(emb_128, mask).squeeze().cpu().item()
        p64 = head_64(emb_64, mask).squeeze().cpu().item()
        pids_list.append(pid)
        names_list.append(name)
        trues.append(overall)
        preds_128.append(p128)
        preds_64.append(p64)

table = pd.DataFrame({
    "player_id": pids_list,
    "имя_игрока": names_list,
    "overall_sofifa": trues,
    "128_рейтинг_предсказание": preds_128,
    "64_рейтинг_предсказание": preds_64,
})
# Усреднение по игроку (один игрок — одна строка)
table_avg = table.groupby("player_id", as_index=False).agg(
    {"имя_игрока": "first", "overall_sofifa": "mean", "128_рейтинг_предсказание": "mean", "64_рейтинг_предсказание": "mean"}
)
table_avg

,player_id,имя_игрока,overall_sofifa,128_рейтинг_предсказание,64_рейтинг_предсказание
0,5458,Santiago Cazorla González,75.0,69.999061,74.871720
1,5459,Santiago Comesaña Veiga,78.0,74.943611,86.530304
2,5472,Sargis Adamyan,66.0,89.024681,77.806152
3,5476,Saviour Gama,54.0,61.833878,61.164970
4,5478,Saîf-Eddine Khaoui,69.0,74.931072,66.693207
...,...,...,...,...,...
219,6381,İrfan Can Kahveci,77.0,68.944077,77.167648
220,6382,İsmail Yüksek,77.0,85.891891,64.997269
221,6383,Ľubomír Šatka,71.0,67.680580,79.745796
222,6384,Łukasz Fabiański,74.0,79.102009,83.512638


In [11]:
# RMSE по каждому игроку (по сезонам одного игрока: sqrt(mean((true - pred)^2)))
import numpy as np

table["se_128"] = (table["overall_sofifa"] - table["128_рейтинг_предсказание"]) ** 2
table["se_64"] = (table["overall_sofifa"] - table["64_рейтинг_предсказание"]) ** 2
rmse_per_player = table.groupby("player_id", as_index=False).agg(
    {"имя_игрока": "first", "se_128": "mean", "se_64": "mean"}
)
rmse_per_player["RMSE_128"] = np.sqrt(rmse_per_player["se_128"])
rmse_per_player["RMSE_64"] = np.sqrt(rmse_per_player["se_64"])
print("Среднее RMSE по игрокам:  128 =", rmse_per_player["RMSE_128"].mean().round(4), "  64 =", rmse_per_player["RMSE_64"].mean().round(4))
rmse_per_player[["player_id", "имя_игрока", "RMSE_128", "RMSE_64"]]

,player_id,имя_игрока,RMSE_128,RMSE_64
0,5458,Santiago Cazorla González,6.383618,6.247850
1,5459,Santiago Comesaña Veiga,3.056389,8.530304
2,5472,Sargis Adamyan,23.024681,11.806152
3,5476,Saviour Gama,7.833878,7.164970
4,5478,Saîf-Eddine Khaoui,5.978367,3.370775
...,...,...,...,...
219,6381,İrfan Can Kahveci,8.055923,0.167648
220,6382,İsmail Yüksek,8.891891,12.002731
221,6383,Ľubomír Šatka,3.319420,8.745796
222,6384,Łukasz Fabiański,6.467456,11.333644
